# Fama-French Factor Replication Research


## Research Design

- **研究对象**：美国股票月度收益、公司市值、账面价值、盈利能力和投资变量。
- **数据来源**：CRSP 月度数据、Compustat 会计数据、Fama-French 官方月度因子。
- **本地数据目录**：`D:\stata\For Students\csv`。
- **三因子复现**：用 size 和 book-to-market 构造 SMB 与 HML。
- **五因子复现**：进一步加入 profitability 和 investment，构造 RMW 与 CMA。
- **评估方法**：将复现因子与官方因子做时间序列回归，观察斜率、解释度和拟合效果。


In [67]:
import pandas as pd
import numpy as np
import sqlite3
import statsmodels.formula.api as smf

try:
    from regtabletotext import prettify_result
except ModuleNotFoundError:
    def prettify_result(model):
        print(model.summary())

import warnings
warnings.filterwarnings("ignore")  # 忽略所有 warning


## 1. Data Loading

读取月度 CRSP、Compustat 以及 Fama-French 官方三因子和五因子数据。


In [68]:
crsp_monthly = pd.read_csv(r"D:\stata\For Students\csv\crsp_monthly.csv")[['permno', 'gvkey', 'date', 'ret_excess', 'mktcap', 'mktcap_lag', 'exchange']]
crsp_monthly['date'] = pd.to_datetime(crsp_monthly['date'])
crsp_monthly=crsp_monthly.dropna()
crsp_monthly.head()


,permno,gvkey,date,ret_excess,mktcap,mktcap_lag,exchange
0,10028,12096.0,1993-03-01,-0.102500,6.329250,7.032500,AMEX
1,10028,12096.0,1993-04-01,0.386489,8.790625,6.329250,AMEX
2,10028,12096.0,1993-05-01,0.197800,10.548750,8.790625,AMEX
3,10028,12096.0,1993-06-01,-0.135833,9.044750,10.548750,AMEX
4,10028,12096.0,1993-07-01,0.189908,10.784125,9.044750,AMEX


In [69]:
compustat = pd.read_csv(r"D:\stata\For Students\csv\compustat.csv")[['gvkey', 'datadate', 'be', 'op', 'inv']]
compustat['datadate'] = pd.to_datetime(compustat['datadate'])
compustat=compustat.dropna()
compustat.head()

,gvkey,datadate,be,op,inv
1951,5410,1961-01-31,14.906,0.182477,-0.050360
1953,6583,1961-01-31,21.845,0.227512,0.028481
1955,1803,1961-01-31,84.535,0.259502,0.117427
1956,7254,1961-01-31,53.500,0.173252,-0.002278
1960,6260,1961-01-31,67.238,0.322288,0.236048


In [70]:
factors_ff3_monthly = pd.read_csv(r"D:\stata\For Students\csv\factors_ff3_monthly.csv")[["date", "smb", "hml"]]
factors_ff3_monthly['date']=pd.to_datetime(factors_ff3_monthly['date'])
factors_ff3_monthly.head()


,date,smb,hml
0,1960-01-01,0.0209,0.0278
1,1960-02-01,0.0051,-0.0193
2,1960-03-01,-0.0049,-0.0294
3,1960-04-01,0.0032,-0.0228
4,1960-05-01,0.0121,-0.0370


In [71]:

factors_ff5_monthly = pd.read_csv(r"D:\stata\For Students\csv\factors_ff5_monthly.csv")[["date", "smb", "hml", "rmw", "cma"]]
factors_ff5_monthly['date']=pd.to_datetime(factors_ff5_monthly['date'])
factors_ff5_monthly.head()


,date,smb,hml,rmw,cma
0,1963-07-01,-0.0041,-0.0097,0.0068,-0.0118
1,1963-08-01,-0.0080,0.0180,0.0036,-0.0035
2,1963-09-01,-0.0052,0.0013,-0.0071,0.0029
3,1963-10-01,-0.0139,-0.0010,0.0280,-0.0201
4,1963-11-01,-0.0088,0.0175,-0.0051,0.0224


## 2. Sorting Variables for Fama-French 3 Factors

三因子复现需要两个核心排序变量：

- **Size**：每年 6 月市值，用于小市值和大市值组合划分。
- **Book-to-market**：账面价值除以年末市值，并在下一年 7 月进入排序。


In [72]:
size = (crsp_monthly
  .query("date.dt.month == 6")
  .assign(sorting_date=lambda x: (x["date"]+pd.DateOffset(months=1)))
  .get(["permno", "exchange", "sorting_date", "mktcap"])
  .rename(columns={"mktcap": "size"})
)
size.head()

,permno,exchange,sorting_date,size
3,10028,AMEX,1993-07-01,9.044750
15,10028,AMEX,1994-07-01,13.209750
27,10028,AMEX,1995-07-01,9.192187
39,10028,AMEX,1996-07-01,8.367688
51,10028,AMEX,1997-07-01,7.735000


In [73]:

market_equity = (crsp_monthly
  .query("date.dt.month == 12")
  .assign(sorting_date=lambda x: (x["date"]+pd.DateOffset(months=7)))
  .get(["permno", "gvkey", "sorting_date", "mktcap"])
  .rename(columns={"mktcap": "me"})
)

In [74]:
book_to_market = (compustat
  .assign(
    sorting_date=lambda x: (pd.to_datetime(
      (x["datadate"].dt.year+1).astype(str)+"0701", format="%Y%m%d")
    )
  )
  .merge(market_equity, how="inner", on=["gvkey", "sorting_date"])
  .assign(bm=lambda x: x["be"]/x["me"])
  .get(["permno", "sorting_date", "me", "bm"])
)
book_to_market.head()

,permno,sorting_date,me,bm
0,19035,1962-07-01,10.93500,1.363146
1,19107,1962-07-01,44.51475,0.490736
2,10559,1962-07-01,200.87200,0.420840
3,22891,1962-07-01,83.22450,0.642840
4,12626,1962-07-01,263.95350,0.254734


In [75]:
sorting_variables = (size
  .merge(book_to_market, how="inner", on=["permno", "sorting_date"])
  .dropna()
  .drop_duplicates(subset=["permno", "sorting_date"])
 )

sorting_variables.head()

,permno,exchange,sorting_date,size,me,bm
0,10028,AMEX,1993-07-01,9.044750,7.735750,0.104967
1,10028,AMEX,1994-07-01,13.209750,13.567125,0.119922
2,10028,AMEX,1995-07-01,9.192187,13.126500,0.133699
3,10028,AMEX,1996-07-01,8.367688,7.287500,0.243431
4,10028,AMEX,1997-07-01,7.735000,6.158250,0.345228


## 3. Portfolio Assignment

使用 NYSE 股票作为断点样本，对 size 做 2 组排序，对 book-to-market 做 3 组排序，形成 2x3 组合。


In [76]:
def assign_portfolio(data, sorting_variable, percentiles):
    """Assign portfolios to a bin according to a sorting variable."""
    
    breakpoints = (data
      .query("exchange == 'NYSE'")
      .get(sorting_variable)
      .quantile(percentiles, interpolation="linear")
    )
    breakpoints.iloc[0] = -np.inf
    breakpoints.iloc[breakpoints.size-1] = np.inf
    
    assigned_portfolios = pd.cut(
      data[sorting_variable],
      bins=breakpoints,
      labels=pd.Series(range(1, breakpoints.size)),
      include_lowest=True,
      right=False
    )
    
    return assigned_portfolios

In [77]:
portfolio_parts = []

for sorting_date, group in sorting_variables.groupby("sorting_date"):
    group = group.copy()
    group["sorting_date"] = sorting_date
    group["portfolio_size"] = assign_portfolio(group, "size", [0, 0.5, 1])
    group["portfolio_bm"] = assign_portfolio(group, "bm", [0, 0.3, 0.7, 1])
    portfolio_parts.append(group)

portfolios = pd.concat(portfolio_parts, ignore_index=True)[[
    "permno", "sorting_date", "portfolio_size", "portfolio_bm"
]]

portfolios.head()


,permno,sorting_date,portfolio_size,portfolio_bm
0,10006,1962-07-01,1,3
1,10102,1962-07-01,2,2
2,10153,1962-07-01,2,3
3,10137,1962-07-01,2,2
4,10161,1962-07-01,2,1


In [78]:
portfolios = (crsp_monthly
  .assign(
    sorting_date=lambda x: (pd.to_datetime(
      x["date"].apply(lambda x: str(x.year-1)+
        "0701" if x.month <= 6 else str(x.year)+"0701")))
  )
  .merge(portfolios, how="inner", on=["permno", "sorting_date"])
)
portfolios.head()

,permno,gvkey,date,ret_excess,mktcap,mktcap_lag,exchange,sorting_date,portfolio_size,portfolio_bm
0,10028,12096.0,1993-03-01,-0.102500,6.329250,7.032500,AMEX,1992-07-01,1,1
1,10028,12096.0,1993-04-01,0.386489,8.790625,6.329250,AMEX,1992-07-01,1,1
2,10028,12096.0,1993-05-01,0.197800,10.548750,8.790625,AMEX,1992-07-01,1,1
3,10028,12096.0,1993-06-01,-0.135833,9.044750,10.548750,AMEX,1992-07-01,1,1
4,10028,12096.0,1993-07-01,0.189908,10.784125,9.044750,AMEX,1993-07-01,1,1


## 4. Replicating Fama-French 3 Factors

在每个组合内计算市值加权收益，然后构造：

- **SMB**：小市值组合平均收益减大市值组合平均收益。
- **HML**：高账面市值比组合平均收益减低账面市值比组合平均收益。


In [79]:
factors_replicated = (portfolios
  .groupby(["portfolio_size", "portfolio_bm", "date"])
  .apply(lambda x: pd.Series({
    "ret": np.average(x["ret_excess"], weights=x["mktcap_lag"])
    })
   )
  .reset_index()
  .groupby("date")
  .apply(lambda x: pd.Series({
    "smb_replicated": (
      x["ret"][x["portfolio_size"] == 1].mean() - 
        x["ret"][x["portfolio_size"] == 2].mean()),
    "hml_replicated": (
      x["ret"][x["portfolio_bm"] == 3].mean() -
        x["ret"][x["portfolio_bm"] == 1].mean())
    }))
  .reset_index()
)
factors_replicated.head()

,date,smb_replicated,hml_replicated
0,1962-07-01,0.004249,-0.034875
1,1962-08-01,0.003261,-0.000131
2,1962-09-01,-0.014792,0.005229
3,1962-10-01,-0.025575,0.000294
4,1962-11-01,0.020368,0.002550


In [80]:
factors_replicated = (factors_replicated
  .merge(factors_ff3_monthly, how="inner", on="date")
  .round(4)
)
factors_replicated.head()

,date,smb_replicated,hml_replicated,smb,hml
0,1962-07-01,0.0042,-0.0349,0.0152,-0.0341
1,1962-08-01,0.0033,-0.0001,0.0122,-0.0120
2,1962-09-01,-0.0148,0.0052,-0.0242,0.0128
3,1962-10-01,-0.0256,0.0003,-0.0397,0.0130
4,1962-11-01,0.0204,0.0026,0.0261,0.0095


## 5. Three-Factor Replication Check

将复现的 SMB/HML 与官方 Fama-French SMB/HML 做时间序列回归。若复现质量较好，回归斜率应接近 1，解释度应较高。


In [81]:
model_smb = (smf.ols(
    formula="smb ~ smb_replicated", 
    data=factors_replicated
  )
  .fit()
)
prettify_result(model_smb)

OLS Model:
smb ~ smb_replicated

Coefficients:
                Estimate  Std. Error  t-Statistic  p-Value
Intercept         -0.000       0.000       -1.288    0.198
smb_replicated     0.994       0.004      229.344    0.000

Summary statistics:
- Number of observations: 726
- R-squared: 0.986, Adjusted R-squared: 0.986
- F-statistic: 52,598.570 on 1 and 724 DF, p-value: 0.000



In [82]:
model_hml = (smf.ols(
    formula="hml ~ hml_replicated", 
    data=factors_replicated
  )
  .fit()
)
prettify_result(model_hml)

OLS Model:
hml ~ hml_replicated

Coefficients:
                Estimate  Std. Error  t-Statistic  p-Value
Intercept          0.000       0.000        1.643    0.101
hml_replicated     0.963       0.007      133.338    0.000

Summary statistics:
- Number of observations: 726
- R-squared: 0.961, Adjusted R-squared: 0.961
- F-statistic: 17,778.904 on 1 and 724 DF, p-value: 0.000



### Three-Factor Replication Conclusion

三因子复现结果整体非常理想，说明基于 CRSP 与 Compustat 数据构造的规模因子和价值因子，与 Fama-French 官方因子高度一致。

- **SMB 复现效果最好**：`smb_replicated` 的回归系数为 **0.994**，非常接近 1；t 值为 **229.344**，p 值为 **0.000**，说明复现 SMB 与官方 SMB 存在极强且显著的线性关系。模型 `R-squared = 0.986`，意味着复现因子可以解释官方 SMB 约 **98.6%** 的波动。
- **HML 复现效果也很强**：`hml_replicated` 的回归系数为 **0.963**，接近 1；t 值为 **133.338**，p 值为 **0.000**，说明复现 HML 与官方 HML 高度相关。模型 `R-squared = 0.961`，解释度达到 **96.1%**。
- 两个回归的截距项都接近 0，且不显著，说明复现因子相对官方因子没有明显系统性偏差。

因此，三因子部分可以认为复现成功。尤其是 SMB 的复现精度接近官方口径，HML 虽略低于 SMB，但拟合度仍然很高，说明 size 与 book-to-market 的排序、组合构造和市值加权方法基本正确。


## 6. Sorting Variables for Fama-French 5 Factors

五因子模型在 size 和 book-to-market 之外，加入：

- **OP**：盈利能力，用于构造 RMW。
- **INV**：投资强度，用于构造 CMA。


In [83]:
other_sorting_variables = (compustat
  .assign(
    sorting_date=lambda x: (pd.to_datetime(
      (x["datadate"].dt.year+1).astype(str)+"0701", format="%Y%m%d")
    )
  )
  .merge(market_equity, how="inner", on=["gvkey", "sorting_date"])
  .assign(bm=lambda x: x["be"]/x["me"])
  .get(["permno", "sorting_date", "me", "bm", "op", "inv"])
)


In [84]:
sorting_variables = (size
  .merge(other_sorting_variables, how="inner", on=["permno", "sorting_date"])
  .dropna()
  .drop_duplicates(subset=["permno", "sorting_date"])
 )
sorting_variables.head()

,permno,exchange,sorting_date,size,me,bm,op,inv
0,10028,AMEX,1993-07-01,9.044750,7.735750,0.104967,-0.125616,-0.227702
1,10028,AMEX,1994-07-01,13.209750,13.567125,0.119922,0.179471,0.667376
2,10028,AMEX,1995-07-01,9.192187,13.126500,0.133699,0.096296,0.135334
3,10028,AMEX,1996-07-01,8.367688,7.287500,0.243431,0.015220,0.103739
4,10028,AMEX,1997-07-01,7.735000,6.158250,0.345228,-0.070555,0.349720


In [85]:
size_parts = []

for sorting_date, group in sorting_variables.groupby("sorting_date"):
    group = group.copy()
    group["sorting_date"] = sorting_date
    group["portfolio_size"] = assign_portfolio(group, "size", [0, 0.5, 1])
    size_parts.append(group)

portfolios_with_size = pd.concat(size_parts, ignore_index=True)

five_factor_parts = []

for (sorting_date, portfolio_size), group in portfolios_with_size.groupby(["sorting_date", "portfolio_size"]):
    group = group.copy()
    group["sorting_date"] = sorting_date
    group["portfolio_size"] = portfolio_size
    group["portfolio_bm"] = assign_portfolio(group, "bm", [0, 0.3, 0.7, 1])
    group["portfolio_op"] = assign_portfolio(group, "op", [0, 0.3, 0.7, 1])
    group["portfolio_inv"] = assign_portfolio(group, "inv", [0, 0.3, 0.7, 1])
    five_factor_parts.append(group)

portfolios = pd.concat(five_factor_parts, ignore_index=True)[[
    "permno", "sorting_date", "portfolio_size", "portfolio_bm", "portfolio_op", "portfolio_inv"
]]

portfolios.head()


,permno,sorting_date,portfolio_size,portfolio_bm,portfolio_op,portfolio_inv
0,10006,1962-07-01,1,3,1,1
1,10188,1962-07-01,1,2,2,1
2,10233,1962-07-01,1,1,3,2
3,10268,1962-07-01,1,2,1,1
4,10460,1962-07-01,1,1,3,2


In [86]:
portfolios = (crsp_monthly
  .assign(
    sorting_date=lambda x: (pd.to_datetime(
      x["date"].apply(lambda x: str(x.year-1)+
        "0701" if x.month <= 6 else str(x.year)+"0701")))
  )
  .merge(portfolios, how="inner", on=["permno", "sorting_date"])
)
portfolios.head()

,permno,gvkey,date,ret_excess,mktcap,mktcap_lag,exchange,sorting_date,portfolio_size,portfolio_bm,portfolio_op,portfolio_inv
0,10028,12096.0,1993-03-01,-0.102500,6.329250,7.032500,AMEX,1992-07-01,1,1,1,1
1,10028,12096.0,1993-04-01,0.386489,8.790625,6.329250,AMEX,1992-07-01,1,1,1,1
2,10028,12096.0,1993-05-01,0.197800,10.548750,8.790625,AMEX,1992-07-01,1,1,1,1
3,10028,12096.0,1993-06-01,-0.135833,9.044750,10.548750,AMEX,1992-07-01,1,1,1,1
4,10028,12096.0,1993-07-01,0.189908,10.784125,9.044750,AMEX,1993-07-01,1,1,1,1


## 7. Replicating Value, Profitability, Investment and Size Factors

分别构造价值因子 HML、盈利因子 RMW、投资因子 CMA，并将三类组合中的 size 溢价合成为五因子模型下的 SMB。


In [87]:
portfolios_value = (portfolios
  .groupby(["portfolio_size", "portfolio_bm", "date"])
  .apply(lambda x: pd.Series({
      "ret": np.average(x["ret_excess"], weights=x["mktcap_lag"])
    })
  )
  .reset_index()
)

In [88]:
factors_value = (portfolios_value
  .groupby("date")
  .apply(lambda x: pd.Series({
    "hml_replicated": (
      x["ret"][x["portfolio_bm"] == 3].mean() - 
        x["ret"][x["portfolio_bm"] == 1].mean())})
  )
  .reset_index()
)
factors_value.head()

,date,hml_replicated
0,1962-07-01,-0.024053
1,1962-08-01,-0.006577
2,1962-09-01,0.004991
3,1962-10-01,-0.002795
4,1962-11-01,0.009973


In [89]:
portfolios_profitability = (portfolios
  .groupby(["portfolio_size", "portfolio_op", "date"])
  .apply(lambda x: pd.Series({
      "ret": np.average(x["ret_excess"], weights=x["mktcap_lag"])
    })
  )
  .reset_index()
)
portfolios_profitability.head()

,portfolio_size,portfolio_op,date,ret
0,1,1,1962-07-01,0.059050
1,1,1,1962-08-01,0.018656
2,1,1,1962-09-01,-0.070744
3,1,1,1962-10-01,-0.030874
4,1,1,1962-11-01,0.128482


In [90]:
factors_profitability = (portfolios_profitability
  .groupby("date")
  .apply(lambda x: pd.Series({
    "rmw_replicated": (
      x["ret"][x["portfolio_op"] == 3].mean() - 
        x["ret"][x["portfolio_op"] == 1].mean())})
  )
  .reset_index()
)
factors_profitability.head()

,date,rmw_replicated
0,1962-07-01,0.020984
1,1962-08-01,0.008683
2,1962-09-01,0.000504
3,1962-10-01,0.012590
4,1962-11-01,-0.007101


In [91]:
portfolios_investment = (portfolios
  .groupby(["portfolio_size", "portfolio_inv", "date"])
  .apply(lambda x: pd.Series({
      "ret": np.average(x["ret_excess"], weights=x["mktcap_lag"])
    })
  )
  .reset_index()
)
portfolios_investment.head()

,portfolio_size,portfolio_inv,date,ret
0,1,1,1962-07-01,0.048590
1,1,1,1962-08-01,0.018437
2,1,1,1962-09-01,-0.068487
3,1,1,1962-10-01,-0.012026
4,1,1,1962-11-01,0.129659


In [92]:
factors_investment = (portfolios_investment
  .groupby("date")
  .apply(lambda x: pd.Series({
    "cma_replicated": (
      x["ret"][x["portfolio_inv"] == 1].mean() - 
        x["ret"][x["portfolio_inv"] == 3].mean())})
  )
  .reset_index()
)
factors_investment.head()

,date,cma_replicated
0,1962-07-01,-0.029187
1,1962-08-01,0.006461
2,1962-09-01,0.002035
3,1962-10-01,0.007685
4,1962-11-01,-0.002213


In [93]:
factors_size = (
  pd.concat(
    [portfolios_value, portfolios_profitability, portfolios_investment], 
    ignore_index=True
  )
  .groupby("date")
  .apply(lambda x: pd.Series({
    "smb_replicated": (
      x["ret"][x["portfolio_size"] == 1].mean() - 
        x["ret"][x["portfolio_size"] == 2].mean())})
  )
  .reset_index()
)
factors_size.head()

,date,smb_replicated
0,1962-07-01,-0.004281
1,1962-08-01,0.003765
2,1962-09-01,-0.012423
3,1962-10-01,-0.026269
4,1962-11-01,0.020659


In [94]:
factors_replicated = (factors_size
  .merge(factors_value, how="outer", on="date")
  .merge(factors_profitability, how="outer", on="date")
  .merge(factors_investment, how="outer", on="date")
)
factors_replicated.head()

,date,smb_replicated,hml_replicated,rmw_replicated,cma_replicated
0,1962-07-01,-0.004281,-0.024053,0.020984,-0.029187
1,1962-08-01,0.003765,-0.006577,0.008683,0.006461
2,1962-09-01,-0.012423,0.004991,0.000504,0.002035
3,1962-10-01,-0.026269,-0.002795,0.012590,0.007685
4,1962-11-01,0.020659,0.009973,-0.007101,-0.002213


In [95]:
factors_replicated = (factors_replicated
  .merge(factors_ff5_monthly, how="inner", on="date")
  .round(4)
)
factors_replicated.head()

,date,smb_replicated,hml_replicated,rmw_replicated,cma_replicated,smb,hml,rmw,cma
0,1963-07-01,-0.0146,-0.0006,0.0006,-0.0096,-0.0041,-0.0097,0.0068,-0.0118
1,1963-08-01,-0.0030,0.0046,0.0022,0.0039,-0.0080,0.0180,0.0036,-0.0035
2,1963-09-01,-0.0078,0.0051,-0.0095,-0.0019,-0.0052,0.0013,-0.0071,0.0029
3,1963-10-01,-0.0079,-0.0151,0.0291,-0.0215,-0.0139,-0.0010,0.0280,-0.0201
4,1963-11-01,-0.0069,0.0084,-0.0057,0.0129,-0.0088,0.0175,-0.0051,0.0224


## 8. Five-Factor Replication Check

将复现的 SMB、HML、RMW、CMA 与官方五因子数据逐一回归，评估各因子的复现效果。


In [96]:
model_smb = (smf.ols(
    formula="smb ~ smb_replicated", 
    data=factors_replicated
  )
  .fit()
)
prettify_result(model_smb)

OLS Model:
smb ~ smb_replicated

Coefficients:
                Estimate  Std. Error  t-Statistic  p-Value
Intercept          -0.00       0.000       -1.490    0.137
smb_replicated      0.97       0.004      221.842    0.000

Summary statistics:
- Number of observations: 714
- R-squared: 0.986, Adjusted R-squared: 0.986
- F-statistic: 49,213.674 on 1 and 712 DF, p-value: 0.000



In [97]:
model_hml = (smf.ols(
    formula="hml ~ hml_replicated", 
    data=factors_replicated
  )
  .fit()
)
prettify_result(model_hml)

OLS Model:
hml ~ hml_replicated

Coefficients:
                Estimate  Std. Error  t-Statistic  p-Value
Intercept          0.000        0.00        1.593    0.112
hml_replicated     0.991        0.01       96.582    0.000

Summary statistics:
- Number of observations: 714
- R-squared: 0.929, Adjusted R-squared: 0.929
- F-statistic: 9,328.089 on 1 and 712 DF, p-value: 0.000



In [98]:
model_rmw = (smf.ols(
    formula="rmw ~ rmw_replicated", 
    data=factors_replicated
  )
  .fit()
)
prettify_result(model_rmw)

OLS Model:
rmw ~ rmw_replicated

Coefficients:
                Estimate  Std. Error  t-Statistic  p-Value
Intercept          0.000       0.000        0.317    0.751
rmw_replicated     0.954       0.009      107.030    0.000

Summary statistics:
- Number of observations: 714
- R-squared: 0.941, Adjusted R-squared: 0.941
- F-statistic: 11,455.356 on 1 and 712 DF, p-value: 0.000



In [99]:
model_cma = (smf.ols(
    formula="cma ~ cma_replicated", 
    data=factors_replicated
  )
  .fit()
)
prettify_result(model_cma)

OLS Model:
cma ~ cma_replicated

Coefficients:
                Estimate  Std. Error  t-Statistic  p-Value
Intercept          0.001       0.000        4.046      0.0
cma_replicated     0.966       0.008      117.723      0.0

Summary statistics:
- Number of observations: 714
- R-squared: 0.951, Adjusted R-squared: 0.951
- F-statistic: 13,858.614 on 1 and 712 DF, p-value: 0.000



### Five-Factor Replication Conclusion

五因子复现结果整体也较好，四个复现因子均与官方因子显著正相关，回归系数均接近 1，说明五因子构造流程基本有效。

- **SMB**：`smb_replicated` 的回归系数为 **0.970**，t 值为 **221.842**，p 值为 **0.000**，`R-squared = 0.986`。说明五因子框架下重新构造的规模因子仍然高度贴近官方 SMB，解释度达到 **98.6%**。
- **HML**：`hml_replicated` 的回归系数为 **0.991**，t 值为 **96.582**，p 值为 **0.000**，`R-squared = 0.929`。说明价值因子方向和幅度都非常接近官方 HML，但解释度低于三因子版本，可能是五因子组合划分更细后样本和权重结构发生变化。
- **RMW**：`rmw_replicated` 的回归系数为 **0.954**，t 值为 **107.030**，p 值为 **0.000**，`R-squared = 0.941`。说明盈利能力因子复现质量较高，OP 排序能够较好捕捉官方 RMW 的变化。
- **CMA**：`cma_replicated` 的回归系数为 **0.966**，t 值为 **117.723**，p 值为 **0.000**，`R-squared = 0.951`。说明投资因子复现效果也较强，INV 排序基本复现了官方 CMA 的主要波动。

综合来看，五因子复现可以认为成功。SMB 的拟合度最高，CMA 和 RMW 次之，HML 在五因子框架下解释度相对最低但仍达到 92.9%。这说明本地数据构造出的五个因子与官方因子高度一致，差异主要可能来自样本筛选、断点计算、会计变量口径和缺失值处理等细节，而不是核心复现逻辑错误。


## Research Summary

本研究完成了 Fama-French 三因子和五因子的核心复现流程：从 CRSP 和 Compustat 数据构造排序变量，按年度形成组合，计算市值加权组合收益，并与官方因子做回归验证。

该 notebook 可以作为量化研究作品集中的因子研究项目，重点展示数据清洗、组合构造、因子复制和结果验证能力。

后续可以进一步扩展：

- 增加复现因子与官方因子的相关系数、均值、波动率和跟踪误差；
- 绘制复现因子与官方因子的累计收益曲线；
- 比较不同断点样本或不同加权方式对复现效果的影响。
